# Harry Potter GPT — full Colab pipeline (Pretrain -> SFT -> HF convert -> DPO)

Re-runs the whole thing end to end on one Colab GPU, since the previous SFT/DPO weights were lost.

**Everything trains onto local Colab disk (`/content/...`), not a mounted Drive path.** An earlier version wrote checkpoints straight into Google Drive during training, but Drive's FUSE mount chokes on the frequent large writes a training loop does (repeated ~1.5GB checkpoint saves) and the run stalled. Instead: train locally (fast, reliable), then after each stage finishes, zip that stage's output folder and copy the single zip file to Drive (one big write, not thousands of small ones). If the runtime disconnects, at worst you lose the stage in progress — everything before it is already a zip sitting in Drive.

Runtime: Runtime -> Change runtime type -> T4 GPU (free tier is fine, this is a 124M model).

Run the cells top to bottom. Each stage prints samples so you can eyeball progress before moving to the next one.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_BACKUP = '/content/drive/MyDrive/harry-potter-gpt'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
print('Drive backup folder ready at', DRIVE_BACKUP)

In [ ]:
!git clone https://github.com/abhijitdalal26/harry-potter-gpt.git /content/harry-potter-gpt
%cd /content/harry-potter-gpt/nanoGPT

## Resume from Drive backups (run this every time, even on a fresh runtime)
Colab wipes local `/content` disk completely on any disconnect or restart — the repo you just cloned is fresh and empty of any prior progress. This cell checks Drive for zips from stages that already finished in an earlier session and restores them locally, so you don't retrain something you already have. Anything not yet in Drive is silently skipped — its training cells further down will just run normally.

In [ ]:
import os

DRIVE_BACKUP = '/content/drive/MyDrive/harry-potter-gpt'
restored = []
for name in ['out-harry-potter', 'out-harry-potter-sft', 'harry-potter-hf', 'harry-potter-hf-dpo']:
    zip_path = f'{DRIVE_BACKUP}/{name}.zip'
    if os.path.exists(zip_path) and not os.path.isdir(name):
        get_ipython().system(f'unzip -oq "{zip_path}" -d .')
        restored.append(name)

if restored:
    print('Restored from Drive (already done — skip that stage\'s training cell, but still fine to rerun its sample/backup cells):')
    for r in restored:
        print(' -', r)
else:
    print('Nothing to restore yet from Drive — run every stage fresh from here.')

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q tiktoken transformers datasets trl accelerate

## Stage 0 — baseline (untouched pretrained GPT-2, no HP training at all)
Sanity check + the "before" reference point for the final comparison.

In [ ]:
!python sample.py --init_from=gpt2 --start="Harry Potter" --num_samples=2 --max_new_tokens=150

## Data prep — tokenize the 7 HP books with the GPT-2 BPE tokenizer
The raw `harry_potter.txt` is already in the repo; the tokenized `.bin` files are gitignored (large + trivial to regenerate).

In [ ]:
!python data/harry_potter/prepare.py

## Stage 1 — Continued pretraining on the HP books
Starts from pretrained GPT-2 (124M), keeps training on the 7 books so it picks up world/characters/style. Saves to the local `out-harry-potter/` folder (the config's own default `out_dir`, no override needed). This folder is kept as a permanent pretrain-only snapshot — Stage 2 will train into a *copy*, not this one, so it survives all the way to the final comparison.

In [ ]:
!python train.py config/finetune_harry_potter.py --compile=False

In [ ]:
!python sample.py --out_dir=out-harry-potter --start="Harry Potter" --num_samples=3 --max_new_tokens=200

In [ ]:
# back up Stage 1 to Drive as a single zip (safety net before moving on)
!zip -rq /content/out-harry-potter.zip out-harry-potter
!cp /content/out-harry-potter.zip /content/drive/MyDrive/harry-potter-gpt/out-harry-potter.zip
print('Backed up out-harry-potter/ (pretrain-only) to Drive')

## Stage 2 — Supervised fine-tuning (SFT) on the chat format
`hp_sft_data.txt` (4,321 lines, `<|user|>`/`<|assistant|>` format) is already in the repo.

**Important:** SFT's config resumes from whatever checkpoint sits in its `out_dir`, and would happily overwrite `out-harry-potter/` in place if pointed at it directly — destroying the pretrain-only checkpoint needed for the final Base-vs-SFT-vs-DPO comparison. So Stage 1's output gets copied into a new folder first, and SFT resumes from and writes into that copy instead, leaving `out-harry-potter/` untouched.

In [ ]:
!cp -r out-harry-potter out-harry-potter-sft
print('Copied Stage 1 checkpoint into out-harry-potter-sft/ for SFT to resume from and write into')

In [ ]:
!python data/harry_potter_sft/prepare_sft.py

In [ ]:
!python train.py config/harry_potter_sft.py --out_dir=out-harry-potter-sft --compile=False

In [ ]:
USER = chr(60) + "|user|" + chr(62)
ASST = chr(60) + "|assistant|" + chr(62)
prompt = f"{USER} Who is Harry Potter?\n{ASST}"
!python sample.py --out_dir=out-harry-potter-sft --start="$prompt" --num_samples=3 --max_new_tokens=200

In [ ]:
!zip -rq /content/out-harry-potter-sft.zip out-harry-potter-sft
!cp /content/out-harry-potter-sft.zip /content/drive/MyDrive/harry-potter-gpt/out-harry-potter-sft.zip
print('Backed up out-harry-potter-sft/ to Drive')

## Stage 3 — Convert the nanoGPT checkpoint to HuggingFace format
Converts the SFT checkpoint (`out-harry-potter-sft/`, not the pretrain-only `out-harry-potter/`) since that's the one DPO needs to build on. Needed so TRL's `DPOTrainer` can use it. The one thing to get right: nanoGPT stores `attn`/`mlp` weights as `Conv1D` (transposed) vs HuggingFace's `nn.Linear` — those four weight matrices get transposed on the way across, everything else copies straight.

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2Config, GPT2Tokenizer

CKPT_PATH  = 'out-harry-potter-sft/ckpt.pt'
OUTPUT_DIR = 'harry-potter-hf'

print("Loading checkpoint...")
ckpt = torch.load(CKPT_PATH, map_location='cpu')
model_args = ckpt['model_args']
state_dict = ckpt['model']
print("Model args:", model_args)

unwanted_prefix = '_orig_mod.'
for k, v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)

key_mapping = {
    'transformer.wte.weight': 'transformer.wte.weight',
    'transformer.wpe.weight': 'transformer.wpe.weight',
    'transformer.ln_f.weight': 'transformer.ln_f.weight',
    'transformer.ln_f.bias': 'transformer.ln_f.bias',
    'lm_head.weight': 'lm_head.weight',
}
for i in range(model_args['n_layer']):
    key_mapping.update({
        f'transformer.h.{i}.ln_1.weight': f'transformer.h.{i}.ln_1.weight',
        f'transformer.h.{i}.ln_1.bias': f'transformer.h.{i}.ln_1.bias',
        f'transformer.h.{i}.ln_2.weight': f'transformer.h.{i}.ln_2.weight',
        f'transformer.h.{i}.ln_2.bias': f'transformer.h.{i}.ln_2.bias',
        f'transformer.h.{i}.attn.c_attn.weight': f'transformer.h.{i}.attn.c_attn.weight',
        f'transformer.h.{i}.attn.c_attn.bias': f'transformer.h.{i}.attn.c_attn.bias',
        f'transformer.h.{i}.attn.c_proj.weight': f'transformer.h.{i}.attn.c_proj.weight',
        f'transformer.h.{i}.attn.c_proj.bias': f'transformer.h.{i}.attn.c_proj.bias',
        f'transformer.h.{i}.mlp.c_fc.weight': f'transformer.h.{i}.mlp.c_fc.weight',
        f'transformer.h.{i}.mlp.c_fc.bias': f'transformer.h.{i}.mlp.c_fc.bias',
        f'transformer.h.{i}.mlp.c_proj.weight': f'transformer.h.{i}.mlp.c_proj.weight',
        f'transformer.h.{i}.mlp.c_proj.bias': f'transformer.h.{i}.mlp.c_proj.bias',
    })

transposed_keys = ['attn.c_attn.weight', 'attn.c_proj.weight', 'mlp.c_fc.weight', 'mlp.c_proj.weight']

new_state_dict = {}
for nano_key, hf_key in key_mapping.items():
    if nano_key in state_dict:
        tensor = state_dict[nano_key]
        if any(nano_key.endswith(k) for k in transposed_keys):
            tensor = tensor.t()
        new_state_dict[hf_key] = tensor
    else:
        print(f"WARNING: missing key {nano_key}")

hf_config = GPT2Config(
    vocab_size=model_args['vocab_size'],
    n_positions=model_args['block_size'],
    n_embd=model_args['n_embd'],
    n_layer=model_args['n_layer'],
    n_head=model_args['n_head'],
    resid_pdrop=0.0, embd_pdrop=0.0, attn_pdrop=0.0, use_cache=True,
)

hf_model = GPT2LMHeadModel(hf_config)
hf_model.load_state_dict(new_state_dict, strict=False)
hf_model.save_pretrained(OUTPUT_DIR)

tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
tokenizer.save_pretrained(OUTPUT_DIR)

print("Saved HF model to", OUTPUT_DIR)

In [ ]:
# quick sanity check the conversion actually worked
model = GPT2LMHeadModel.from_pretrained(OUTPUT_DIR).cuda().eval()
tok = GPT2Tokenizer.from_pretrained(OUTPUT_DIR)
prompt = f"{USER} Who is Harry Potter?\n{ASST}"
ids = tok(prompt, return_tensors='pt').to('cuda')
out = model.generate(**ids, max_new_tokens=150, do_sample=True, temperature=0.8, top_p=0.9, top_k=50, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=False))

In [ ]:
!zip -rq /content/harry-potter-hf.zip harry-potter-hf
!cp /content/harry-potter-hf.zip /content/drive/MyDrive/harry-potter-gpt/harry-potter-hf.zip
print('Backed up harry-potter-hf/ to Drive')

## Stage 4 — DPO (preference alignment)
`dpo_data.json` (347 chosen/rejected pairs) is already in the repo. `DPOTrainer` trains the SFT model to prefer the fan-toned response over the flat/generic one, directly, no separate reward model.

In [ ]:
!python data/harry_potter_dpo/prepare_dpo.py

In [ ]:
%%writefile train_dpo_colab.py
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from datasets import load_from_disk
from trl import DPOTrainer, DPOConfig

MODEL_DIR = "harry-potter-hf"
DPO_DATA_DIR = "data/harry_potter_dpo"
OUTPUT_DIR = "harry-potter-hf-dpo"

model = GPT2LMHeadModel.from_pretrained(MODEL_DIR)
tokenizer = GPT2Tokenizer.from_pretrained(MODEL_DIR)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

ref_model = GPT2LMHeadModel.from_pretrained(MODEL_DIR)

train_dataset = load_from_disk(f"{DPO_DATA_DIR}/train")
eval_dataset = load_from_disk(f"{DPO_DATA_DIR}/val")
print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

dpo_config = DPOConfig(
    output_dir=OUTPUT_DIR,
    beta=0.1,
    learning_rate=5e-6,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    max_length=384,
    max_prompt_length=192,
    warmup_ratio=0.1,
    logging_steps=10,
    save_steps=50,
    eval_steps=50,
    eval_strategy="steps",
    save_strategy="steps",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    remove_unused_columns=False,
    report_to="none",
    gradient_checkpointing=True,
    precompute_ref_log_probs=True,
)

trainer = DPOTrainer(
    model=model, ref_model=ref_model, args=dpo_config,
    train_dataset=train_dataset, eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print("DPO model saved to", OUTPUT_DIR)

In [ ]:
# NOTE: if this errors on DPOConfig/DPOTrainer kwargs, TRL's API has likely drifted
# since this was written (mid-2026) — check `python -c "import trl; print(trl.__version__)"`
# and drop/rename whichever kwarg it complains about.
!python train_dpo_colab.py

In [ ]:
!zip -rq /content/harry-potter-hf-dpo.zip harry-potter-hf-dpo
!cp /content/harry-potter-hf-dpo.zip /content/drive/MyDrive/harry-potter-gpt/harry-potter-hf-dpo.zip
print('Backed up harry-potter-hf-dpo/ to Drive')

## Final showcase — Base vs SFT vs DPO, basic -> advanced questions
Same question set run against all three stages, all loaded from local disk. Saves to a local markdown file, which the last cell zips up with everything else.

In [ ]:
import torch, os
from transformers import GPT2LMHeadModel, GPT2Tokenizer
from model import GPTConfig, GPT
import tiktoken

os.makedirs('results', exist_ok=True)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# base (nanoGPT format, true pretrain-only checkpoint — out-harry-potter/ was never touched by SFT)
ckpt = torch.load('out-harry-potter/ckpt.pt', map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
base_model = GPT(gptconf)
sd = ckpt['model']
for k in list(sd.keys()):
    if k.startswith('_orig_mod.'):
        sd[k[len('_orig_mod.'):]] = sd.pop(k)
base_model.load_state_dict(sd)
base_model.eval().to(device)
enc = tiktoken.get_encoding('gpt2')

# SFT + DPO (HF format)
tok = GPT2Tokenizer.from_pretrained('harry-potter-hf')
tok.pad_token = tok.eos_token
sft_model = GPT2LMHeadModel.from_pretrained('harry-potter-hf').to(device).eval()
dpo_model = GPT2LMHeadModel.from_pretrained('harry-potter-hf-dpo').to(device).eval()

USER = chr(60) + "|user|" + chr(62)
ASST = chr(60) + "|assistant|" + chr(62)

# basic -> advanced, mirrors the DPO training data's own category spread
questions = [
    "Who is Harry Potter?",
    "What is the Mirror of Erised?",
    "Why did Voldemort fail to kill Harry as a baby?",
    "Is Dumbledore a good person?",
    "Does Snape try to save Harry Potter?",
    "Why didn't they just use Veritaserum on Sirius to prove he was innocent?",
    "I just finished Prisoner of Azkaban and I am destroyed about Sirius",
    "I have a theory that Dumbledore is actually Death from the Three Brothers tale",
]

def gen_base(prompt, max_new_tokens=150):
    ids = enc.encode(prompt, allowed_special={"<|endoftext|>"})
    x = torch.tensor(ids, dtype=torch.long, device=device)[None, ...]
    with torch.no_grad():
        y = base_model.generate(x, max_new_tokens, temperature=0.8, top_k=50)
    return enc.decode(y[0].tolist()[len(ids):]).strip()

def gen_hf(model, prompt, max_new_tokens=150):
    inp = tok(prompt, return_tensors='pt').to(device)
    n = inp['input_ids'].shape[1]
    with torch.no_grad():
        out = model.generate(**inp, max_new_tokens=max_new_tokens, do_sample=True,
                              temperature=0.8, top_p=0.9, top_k=50, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][n:], skip_special_tokens=True).strip()

lines = ["# Harry Potter GPT — stage comparison transcript\n"]
for q in questions:
    prompt = f"{USER} {q}\n{ASST}"
    base_ans = gen_base(prompt)
    sft_ans = gen_hf(sft_model, prompt)
    dpo_ans = gen_hf(dpo_model, prompt)
    block = f"\n## Q: {q}\n\n**Base (pretrain only):** {base_ans}\n\n**SFT:** {sft_ans}\n\n**DPO:** {dpo_ans}\n"
    print(block)
    lines.append(block)

with open('results/stage_comparison.md', 'w', encoding='utf-8') as f:
    f.write('\n'.join(lines))
print("\n\nSaved local transcript to results/stage_comparison.md")

## Final step — zip everything together and push once to Drive
One archive containing all four stages' outputs plus the comparison transcript. This is the only large upload of the whole run — everything before it trained and lived on local disk.

In [ ]:
%cd /content/harry-potter-gpt/nanoGPT
!zip -rq /content/harry-potter-gpt-full.zip out-harry-potter out-harry-potter-sft harry-potter-hf harry-potter-hf-dpo results
!cp /content/harry-potter-gpt-full.zip /content/drive/MyDrive/harry-potter-gpt/harry-potter-gpt-full.zip
!cp results/stage_comparison.md /content/drive/MyDrive/harry-potter-gpt/stage_comparison.md
print('Done — harry-potter-gpt-full.zip and stage_comparison.md are in Drive under harry-potter-gpt/')

## Done

Everything of value now lives in Google Drive under `harry-potter-gpt/`:
- `out-harry-potter.zip` — Stage 1 nanoGPT checkpoint (pretrain-only, untouched by SFT)
- `out-harry-potter-sft.zip` — Stage 2 nanoGPT checkpoint (SFT, chat format)
- `harry-potter-hf.zip` — Stage 3, the SFT model converted to HuggingFace format
- `harry-potter-hf-dpo.zip` — Stage 4 DPO model (HuggingFace format)
- `stage_comparison.md` — the basic-to-advanced transcript across all three stages, plain text, no unzip needed
- `harry-potter-gpt-full.zip` — all of the above bundled together

Next: share `stage_comparison.md` back — that's what turns into the real conversation examples on the portfolio page. If you want to host a live demo, `harry-potter-hf-dpo.zip` (~474MB) is what gets uploaded to Hugging Face.